# 35회 후기 유형 연습 해설

공식 원문·정답이 아닌 창작 합성 자료입니다. README와 출처·보완 명세를 먼저 확인하세요. Python 3.12 별도 실행 환경을 사용합니다. 아래 문제 지문은 PDF와 같으며 설명 셀의 선택 근거·가정·가설과 실행 출력의 계산값·기각 여부·상황 결론을 함께 읽으세요. `분석코드.py`로 같은 계산을 일괄 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys
FOLDER = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
repo = next(p for p in [FOLDER]+list(FOLDER.parents) if (p/'tools/exams').is_dir())
sys.path.insert(0,str(repo/'기출문제/유형연습/공통'))
from analysis import Context
ctx=Context(FOLDER)


## 1. 위치를 이용한 관측소 자료 연결

확인 근거가 제한되어 35회 전체가 아닌 공간 전처리 유형만 재구성했다. rides.csv는 날짜·station_id·rides, stations.csv는 역 위경도, weather_geo.csv는 date·sensor_id·위경도·temp다. 좌표 단위는 도, 구면 반지름은 6371km다. 같은 거리면 sensor_id 사전순을 택한다.

### 1-1

**문제** 위경도 범위·날짜·키 중복·관측소 날짜 커버리지를 확인하시오. 역과 관측소가 서로 다른 키를 가질 때 필요한 연결 기준을 설명하시오.

**풀이·선택 근거·해석**

위도 [-90,90],경도 [-180,180]과 중복을 검증한다. 날짜만으로 many-to-many 결합하면 행이 증식하므로 공간 매핑을 먼저 고정한다.

In [2]:
ctx.run('1-1', 'geo_validate')


=== 1-1 ===
input shapes
{
  "rides": [
    36,
    3
  ],
  "stations": [
    3,
    3
  ],
  "weather": [
    36,
    5
  ]
}


### 1-2

**문제** 두 위경도 사이 구면 거리를 계산해 각 역의 가장 가까운 관측소를 찾으시오. 도/라디안·단위·동률 처리와 동일 좌표 거리 0을 검증하시오.

**풀이·선택 근거·해석**

Haversine을 radian으로 계산하고 구면반지름을 km로 둔다. 부동소수 때문에 삼각함수 입력을 [0,1]로 제한한다. 가까운 좌표 검색을 무조건 직교 거리로 대체하지 않는다.

In [3]:
ctx.run('1-2', 'geo_map')


=== 1-2 ===
nearest sensor
[
  {
    "station_id": "S1",
    "sensor_id": "W1",
    "distance_km": 1.4362947839864137
  },
  {
    "station_id": "S2",
    "sensor_id": "W2",
    "distance_km": 1.4360117629390292
  },
  {
    "station_id": "S3",
    "sensor_id": "W3",
    "distance_km": 1.43650692045423
  }
]


### 1-3

**문제** 매핑을 이용해 역별 일별 이용량에 기상을 연결하시오. 일대일 키·원래 행 수 보존·미매칭 여부를 검증하고 관측소 변경·기상 가용 시점·공간 대표성의 한계를 설명하시오.

**풀이·선택 근거·해석**

고정 station→sensor 매핑 후 date,sensor_id로 결합한다. 더 가까워도 해당 날짜 관측이 없을 수 있어 결합 지시자로 누락을 확인한다. 예측 모형·통계 문제를 확인하지 못했으므로 덧붙여 복원하지 않는다.

In [4]:
ctx.run('1-3', 'geo_join')


=== 1-3 ===
joined
{
  "rows": 36,
  "unmatched": 0,
  "preview": [
    {
      "index": 0,
      "date": "2023-01-01",
      "station_id": "S1",
      "rides": 336,
      "sensor_id": "W1",
      "distance_km": 1.4362947839864137,
      "lat": 35.15,
      "lon": 126.84,
      "temp": 14.9785258,
      "_merge": "both"
    },
    {
      "index": 1,
      "date": "2023-01-01",
      "station_id": "S2",
      "rides": 135,
      "sensor_id": "W2",
      "distance_km": 1.4360117629390292,
      "lat": 35.2,
      "lon": 126.91,
      "temp": 11.3612575,
      "_merge": "both"
    },
    {
      "index": 2,
      "date": "2023-01-01",
      "station_id": "S3",
      "rides": 210,
      "sensor_id": "W3",
      "distance_km": 1.43650692045423,
      "lat": 35.13,
      "lon": 126.93,
      "temp": 16.675492,
      "_merge": "both"
    },
    {
      "index": 3,
      "date": "2023-01-02",
      "station_id": "S1",
      "rides": 218,
      "sensor_id": "W1",
      "distance_km": 1.436294